# 04 — Descriptor-dimension selection

This notebook reads the manually selected layer set from Notebook 03 and keeps
that set fixed. It compares the primary Uniform layer weighting global-local
head at 128-D and 256-D using SfM validation only. It writes the final-model
lock consumed by Notebook 05.


In [ ]:
# Run this first in every fresh Colab runtime. It intentionally does not use PYTHONPATH.
from datetime import datetime, timezone
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get('CBIR_REPO_URL', 'https://github.com/armin-faraji/LightweightCBIR.git')
REPO_REVISION = os.environ.get('CBIR_REPO_REVISION', 'main')
PROJECT_ROOT = Path('/content/lightweight-cbir')
if not PROJECT_ROOT.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REVISION], check=True)
os.chdir(PROJECT_ROOT)
required_project_files = (
    PROJECT_ROOT / 'pyproject.toml',
    PROJECT_ROOT / 'src' / 'cbir' / '__init__.py',
    PROJECT_ROOT / 'src' / 'cbir' / 'artifacts.py',
)
missing_project_files = [
    str(path.relative_to(PROJECT_ROOT)) for path in required_project_files if not path.is_file()
]
if missing_project_files:
    raise RuntimeError(
        'The cloned repository does not contain the required project code: '
        + ', '.join(missing_project_files)
    )
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py', 'scipy', 'PyYAML', 'tqdm', 'matplotlib', 'Pillow'], check=True)

importlib.invalidate_caches()
import cbir
print('cbir package:', cbir.__file__)

from cbir.artifacts import create_artifact_run, make_artifact_run_id
from cbir.cloud import mount_colab_drive, runtime_report, write_runtime_report
from cbir.config import config_to_dict, load_project_config
from cbir.utils import stable_hash

PERSISTENT_ROOT = mount_colab_drive() / 'lightweight-cbir'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TORCH_HOME', str(PERSISTENT_ROOT / 'torch_hub'))
CONFIG_PATH = Path('configs/colab.yaml')
cfg = load_project_config(CONFIG_PATH)
environment = runtime_report(project_root=PROJECT_ROOT)
config_fingerprint = stable_hash(config_to_dict(cfg))
RUN_ID = make_artifact_run_id(
    notebook='04', git_sha=environment['git_sha'], config_fingerprint=config_fingerprint
)
ARTIFACT_RUN = create_artifact_run(
    PROJECT_ROOT / 'outputs',
    '04',
    run_id=RUN_ID,
    metadata={'git_sha': environment['git_sha'], 'config_fingerprint': config_fingerprint},
)
LOCAL_OUTPUT_DIR = ARTIFACT_RUN.local_dir
DRIVE_OUTPUT_ROOT = PERSISTENT_ROOT / 'notebook_outputs'
write_runtime_report(
    LOCAL_OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    extra={'notebook': '04', 'config': str(CONFIG_PATH), 'stage': 'descriptor_dimension_selection'},
)
shutil.copy2(CONFIG_PATH, ARTIFACT_RUN.path_for('config.yaml'))
ARTIFACT_RUN.write_json('effective_config.json', config_to_dict(cfg))
print('Project:', PROJECT_ROOT)
print('Artifacts:', LOCAL_OUTPUT_DIR)
print('Descriptor-dimension selection runtime ready.')


## Restore the manual layer selection, metadata, and feature cache


In [ ]:
from dataclasses import replace

from cbir.cache import FeatureShardReader
from cbir.cloud import publish_file, stage_file
from cbir.config import config_to_dict, train_fingerprint
from cbir.data.sfm import Sfm30kMetadata
from cbir.evaluation import evaluate_sfm_verified_pairs, final_cls_descriptors_from_cache
from cbir.fusion import build_descriptor_head
from cbir.plotting import HorizontalReference, SeriesData, plot_series
from cbir.training import HeadTrainer
from cbir.utils import atomic_write_json, read_json, seed_everything
from cbir.workflow import restore_complete_sfm_cache

LAYER_SELECTION_PATH = PERSISTENT_ROOT / 'selected' / 'layer_selection.json'
if not LAYER_SELECTION_PATH.is_file():
    raise FileNotFoundError('Notebook 03 has not recorded a manual layer selection: ' + str(LAYER_SELECTION_PATH))
layer_selection = read_json(LAYER_SELECTION_PATH)
if layer_selection.get('selected_on') != 'Manual student decision after SfM-30k validation inspection only':
    raise ValueError('Refusing a layer-selection record without the required SfM-only manual-selection provenance.')
SELECTED_LAYERS_ONE_BASED = tuple(int(layer) for layer in layer_selection['selected_layer_indices_one_based'])

LOCAL_SFM_ROOT = cfg.sfm.metadata_path.parent
DRIVE_SFM_ROOT = PERSISTENT_ROOT / 'datasets' / 'sfm30k'
metadata_files = (cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
LOCAL_SFM_ROOT.mkdir(parents=True, exist_ok=True)
if not all(path is not None and path.is_file() for path in metadata_files):
    if not all(path is not None and (DRIVE_SFM_ROOT / path.name).is_file() for path in metadata_files):
        raise FileNotFoundError('SfM metadata is absent locally and on Drive; run Notebook 02 first.')
    for path in metadata_files:
        assert path is not None
        stage_file(DRIVE_SFM_ROOT / path.name, path)
if cfg.sfm.names_clusters_path is None:
    raise ValueError('configs/colab.yaml must define sfm.names_clusters_path')
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
cache_location = restore_complete_sfm_cache(cfg, metadata)
reader = FeatureShardReader(cache_location.local_dir)
reader.max_cached_shards = len(reader.manifest.shards)
if layer_selection.get('cache_fingerprint') != reader.manifest.fingerprint:
    raise ValueError('Layer-selection record does not match the restored frozen cache.')
val_ids = metadata.image_ids('val')
val_cases = metadata.build_validation_cases()
baseline = final_cls_descriptors_from_cache(reader, val_ids)
baseline_report = evaluate_sfm_verified_pairs(baseline, val_ids, val_cases)
baseline_metrics = {
    'label': 'Final CLS — 384-D (frozen)',
    'descriptor_dimension': 384,
    'recall_at_1': baseline_report.recall_at_1,
    'recall_at_5': baseline_report.recall_at_5,
    'recall_at_10': baseline_report.recall_at_10,
    'mrr': baseline_report.mrr,
}
ARTIFACT_RUN.write_json('layer_selection_input.json', layer_selection)
ARTIFACT_RUN.write_json('baseline_metrics.json', baseline_metrics)
print('Selected layer set:', SELECTED_LAYERS_ONE_BASED)
print('Using cache:', cache_location.local_dir)

DIMENSION_SPECS = (
    {
        'name': 'uniform_layer_weighting_128',
        'label': 'Uniform layer weighting — 128-D',
        'output_dim': 128,
    },
    {
        'name': 'uniform_layer_weighting_256',
        'label': 'Uniform layer weighting — 256-D',
        'output_dim': 256,
    },
)
PERSISTENT_CHECKPOINT_ROOT = PERSISTENT_ROOT / 'checkpoints' / '04' / ARTIFACT_RUN.run_id
histories = {}
dimension_results = {}


def selected_zero_based_layers():
    return tuple(layer - 1 for layer in SELECTED_LAYERS_ONE_BASED)


def write_dimension_summary():
    ARTIFACT_RUN.write_json('dimension_summary.json', {
        'layer_selection': layer_selection,
        'baseline': baseline_metrics,
        'experiments': dimension_results,
    })


def run_dimension_experiment(name):
    if name in dimension_results:
        return histories[name], dimension_results[name]
    spec = next(item for item in DIMENSION_SPECS if item['name'] == name)
    fusion_cfg = replace(
        cfg.fusion,
        layer_indices=selected_zero_based_layers(),
        output_dim=spec['output_dim'],
        head_kind='global_local',
        gate_mode='uniform',
    )
    run_fingerprint = train_fingerprint(
        cache_fingerprint=reader.manifest.fingerprint,
        fusion=fusion_cfg,
        training=cfg.training,
    )
    local_checkpoint_dir = ARTIFACT_RUN.path_for('checkpoints' / name)
    persistent_checkpoint = PERSISTENT_CHECKPOINT_ROOT / name / 'best.pt'
    seed_everything(cfg.training.seed)
    head = build_descriptor_head(fusion_cfg)
    trainable_parameter_count = sum(parameter.numel() for parameter in head.parameters())
    trainer = HeadTrainer(
        head=head,
        reader=reader,
        train_pairs=metadata.train_pairs,
        fusion_config=fusion_cfg,
        training_config=cfg.training,
        validation_cases=val_cases,
        validation_image_ids=val_ids,
        output_dir=local_checkpoint_dir,
        checkpoint_callback=lambda path: publish_file(path, persistent_checkpoint),
    )
    history = trainer.fit()
    if history.best_checkpoint is None or history.best_epoch is None:
        raise RuntimeError('dimension experiment did not produce a checkpoint')
    result = {
        'experiment_label': spec['label'],
        'layer_indices_one_based': list(SELECTED_LAYERS_ONE_BASED),
        'descriptor_dimension': spec['output_dim'],
        'trainable_parameter_count': trainable_parameter_count,
        'descriptor_bytes_fp32': 4 * spec['output_dim'],
        'descriptor_bytes_fp16': 2 * spec['output_dim'],
        'fusion_config': config_to_dict(fusion_cfg),
        'training_config': config_to_dict(cfg.training),
        'run_fingerprint': run_fingerprint,
        'best_epoch': history.best_epoch + 1,
        'best_metric': history.best_metric,
        'best_epoch_metrics': history.epochs[history.best_epoch],
        'local_checkpoint': str(history.best_checkpoint),
        'persistent_checkpoint': str(persistent_checkpoint),
    }
    histories[name] = history
    dimension_results[name] = result
    write_dimension_summary()
    print(f"{spec['label']}: best SfM validation R@1={history.best_metric:.4f} at epoch {history.best_epoch + 1}")
    return history, result

write_dimension_summary()


## Uniform layer weighting — 128-D


In [ ]:
run_dimension_experiment('uniform_layer_weighting_128')


## Uniform layer weighting — 256-D


In [ ]:
run_dimension_experiment('uniform_layer_weighting_256')


## Compare the fixed-layer descriptor dimensions


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

missing = [spec['name'] for spec in DIMENSION_SPECS if spec['name'] not in dimension_results]
if missing:
    raise RuntimeError('Run both descriptor dimensions before comparing: ' + ', '.join(missing))

series = {
    result['experiment_label']: SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in histories[name].epochs],
        y=[epoch['val_recall_at_1'] for epoch in histories[name].epochs],
    )
    for name, result in dimension_results.items()
}
figure, _ = plot_series(
    series,
    title='SfM validation R@1 during descriptor-dimension selection',
    xlabel='Epoch',
    ylabel='R@1',
    legend_title='Descriptor dimension',
    horizontal_references={
        baseline_metrics['label']: HorizontalReference(baseline_metrics['recall_at_1'])
    },
    fig_size=(11, 6),
    save_path=ARTIFACT_RUN.path_for('figures/dimension_selection_validation_r1.png'),
)
display(figure)
plt.close(figure)
write_dimension_summary()
print({name: result['best_metric'] for name, result in dimension_results.items()})


## Manually lock the final SfM-selected model

Set the variable in the next cell only after inspecting the SfM-only dimension
comparison. Notebook 05 refuses to run without this final-model lock.


In [ ]:
from cbir.cache import sha256_file
from cbir.utils import atomic_write_json

# Select manually after inspecting SfM-only dimension results. Example:
# FINAL_EXPERIMENT = 'uniform_layer_weighting_128'
FINAL_EXPERIMENT = None
if FINAL_EXPERIMENT is None:
    print('No final model locked yet. Set FINAL_EXPERIMENT after inspecting SfM-only results.')
else:
    if FINAL_EXPERIMENT not in dimension_results:
        raise KeyError('Choose a completed dimension experiment: ' + repr(tuple(dimension_results)))
    selected = dimension_results[FINAL_EXPERIMENT]
    checkpoint_path = Path(selected['persistent_checkpoint'])
    if not checkpoint_path.is_file():
        raise FileNotFoundError('Durable selected checkpoint is missing: ' + str(checkpoint_path))
    final_model = {
        'selected_on': 'Manual student decision after SfM-30k validation inspection only',
        'dimension_selection_notebook_run_id': ARTIFACT_RUN.run_id,
        'layer_selection_notebook_run_id': layer_selection['notebook_run_id'],
        'cache_fingerprint': reader.manifest.fingerprint,
        'selected_experiment': FINAL_EXPERIMENT,
        'experiment_label': selected['experiment_label'],
        'layer_indices_one_based': selected['layer_indices_one_based'],
        'descriptor_dimension': selected['descriptor_dimension'],
        'fusion_config': selected['fusion_config'],
        'training_config': selected['training_config'],
        'best_epoch': selected['best_epoch'],
        'best_metric': selected['best_metric'],
        'checkpoint_path': str(checkpoint_path),
        'checkpoint_sha256': sha256_file(checkpoint_path),
    }
    final_lock_path = PERSISTENT_ROOT / 'locked' / 'final_model.json'
    atomic_write_json(final_lock_path, final_model)
    ARTIFACT_RUN.write_json('final_model.json', final_model)
    print('Locked final SfM-selected model:', checkpoint_path)


## Publish Notebook 04 outputs to Drive


In [ ]:
published_output = ARTIFACT_RUN.publish(DRIVE_OUTPUT_ROOT)
print('Validated notebook artifacts published to:', published_output)
